# Flight Delay Detective
## An AI-Powered Flight Analytics & Prediction System

**Dataset:** Kaggle — Flight Delay Dataset 2024  
**Task:** Binary classification — predict whether a completed, non-cancelled, non-diverted flight will arrive 15 or more minutes late  
**Target:** `is_delayed = 1` if `arr_delay >= 15`, else `0`  

---
### Project Components
1. Data processing (chunked CSV reader, filters, feature engineering)
2. Exploratory Data Analysis (EDA)
3. ML pipeline — model comparison and selection
4. FastAPI backend (`backend/main.py`)
5. Frontend dashboard (`frontend/index.html`)
6. Grounded GenAI analyst (`backend/analyst.py`)

## 1. Dataset Overview

| Property | Value |
|---|---|
| File | `flight_data_2024.csv` |
| File size | 1,248.4 MB (~1.25 GB) |
| Raw rows | **7,079,081** |
| Columns | **35** |
| Year coverage | 2024 (all 12 months) |

**Filtering applied:**
- Remove `cancelled = 1` (~1.36% of flights)
- Remove `diverted = 1`
- Remove rows where `arr_delay` is null
- **Valid rows after filtering: 6,965,267**

In [ ]:
# Run data processing pipeline
# This reads the full CSV in 200,000-row chunks and saves a 5% dev sample
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'src/data/processing.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
import json
from pathlib import Path

meta = json.loads(Path('data/processed/metadata.json').read_text())
print('Raw CSV rows         :', f"{meta['raw_row_count']:,}")
print('Valid rows (filtered):', f"{meta['valid_row_count']:,}")
print('Dev sample rows (5%) :', f"{meta['dev_sample_row_count']:,}")
print('Delayed flights      :', f"{meta['n_delayed']:,}  ({meta['pct_delayed']}%)")
print('Not delayed          :', f"{meta['n_not_delayed']:,}  ({100-meta['pct_delayed']}%)")
print('Missing values       :', meta['missing_after_preprocessing'] or 'none')

### Actual results from processing run

| Metric | Value |
|---|---|
| Raw CSV rows | 7,079,081 |
| Valid rows after filtering | 6,965,267 |
| Development subset (5%) | **348,266** |
| Delayed (is_delayed=1) | 72,499 (20.82%) |
| Not delayed (is_delayed=0) | 275,767 (79.18%) |
| Missing values after preprocessing | **none** |

## 2. Approved Pre-Flight Features

All features are available **before** the flight departs — no post-event leakage.

| Feature | Type | Notes |
|---|---|---|
| `op_unique_carrier` | Categorical | Airline IATA code — target encoded |
| `origin` | Categorical | Origin airport IATA code — target encoded |
| `dest` | Categorical | Destination airport — target encoded |
| `month` | Numerical | 1–12, seasonal signal |
| `day_of_month` | Numerical | 1–31 |
| `day_of_week` | Numerical | 1=Mon … 7=Sun |
| `dep_hour` | Derived | `crs_dep_time // 100`, range 0–23 |
| `arr_hour` | Derived | `crs_arr_time // 100`, range 0–23 |
| `crs_elapsed_time` | Numerical | Scheduled duration (minutes) |
| `distance` | Numerical | Route distance (miles) |

**Excluded (post-event / leakage):** `dep_delay`, `dep_time`, `taxi_out`, `wheels_off`, `wheels_on`, `taxi_in`, `arr_time`, `actual_elapsed_time`, `air_time`, `carrier_delay`, `weather_delay`, `nas_delay`, `security_delay`, `late_aircraft_delay`, `arr_delay` (target source)

## 3. Exploratory Data Analysis

In [ ]:
# Run EDA — generates 6 plots and data/analytics/eda_stats.json
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'src/eda/eda.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
from IPython.display import Image, display
import ipywidgets as widgets

figures = [
    ('reports/figures/01_target_distribution.png',       'Fig 1 — Class Distribution'),
    ('reports/figures/02_delay_rate_by_carrier.png',     'Fig 2 — Delay Rate by Carrier'),
    ('reports/figures/03_delay_rate_by_month.png',       'Fig 3 — Delay Rate by Month'),
    ('reports/figures/04_delay_rate_by_dep_hour.png',    'Fig 4 — Delay Rate by Departure Hour'),
    ('reports/figures/05_delay_rate_by_origin_top20.png','Fig 5 — Top-20 Origin Airports'),
    ('reports/figures/06_delay_rate_by_distance_bin.png','Fig 6 — Distance vs Delay'),
]
for path, caption in figures:
    print(f'\n{caption}')
    display(Image(filename=path))

### EDA Key Findings (actual computed values)

| Finding | Value |
|---|---|
| Overall delay rate | **20.82%** |
| Worst month | July — **29.62%** delay rate |
| Best month | October — **13.20%** delay rate |
| Best departure hour | Hour 6 — **9.59%** delay rate |
| Worst departure hour | Hour 20 — **30.59%** delay rate |
| Worst carrier | F9 (Frontier) — **28.64%** |
| Best carrier | YX (Midwest) — **14.45%** |
| Worst origin airport (top-20 busiest) | DFW — **27.41%** |
| Best origin airport (top-20 busiest) | SLC — **16.74%** |

## 4. Machine Learning Pipeline

### 4.1 Chronological Split

| Split | Months | Rows | % Delayed |
|---|---|---|---|
| Train | Jan–Sep (1–9) | 260,001 | 22.35% |
| Validation | Oct–Nov (10–11) | 59,023 | 13.91% |
| Test | Dec (12) | 29,242 | — (untouched) |

**Why chronological?** Random splits allow future seasonal data to leak into training. Chronological splitting replicates the real deployment scenario: training on past data, predicting future flights.

### 4.2 Leakage-Safe Target Encoding

Categorical features (`op_unique_carrier`, `origin`, `dest`) are encoded using **5-fold out-of-fold (OOF) smoothed target encoding**:
- For each training fold, the encoding is computed from the **other 4 folds only**
- Smoothing formula: `encoded = (n * mean_cat + 20 * global_mean) / (n + 20)`
- Validation and test receive encodings from the full training set — no label leakage
- Global mean fallback for unseen categories at inference

In [ ]:
# Run model comparison experiment
# Trains 3 RF configs + 2 GBT configs, selects best by ROC-AUC
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'src/ml/experiment.py'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3000 chars (comparison table + selection)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

### 4.3 Model Comparison Results (Validation Set: Oct–Nov)

| Model | ROC-AUC | Recall | Precision | F1 | Accuracy |
|---|---|---|---|---|---|
| RF baseline (OOF enc.) | 61.75% | 25.46% | 22.38% | 23.82% | 77.34% |
| RF deeper (n=400, d=30) | 61.68% | 23.23% | 22.93% | 23.08% | 78.46% |
| RF shallow (d=12, leaf=100) | 61.73% | 28.95% | 21.57% | 24.72% | 75.47% |
| GBT (n=200, d=5, lr=0.1) | 62.04% | 28.51% | 22.52% | 25.16% | 76.41% |
| **GBT (n=300, d=6, lr=0.05)** ✓ | **62.13%** | **27.80%** | **22.21%** | **24.69%** | **76.41%** |

**Selected model:** `GBT_deeper_slower` — highest ROC-AUC (62.13%) with competitive recall

### 4.4 Confusion Matrix (Validation — Oct/Nov)

| | Predicted: Not Delayed | Predicted: Delayed |
|---|---|---|
| **Actual: Not Delayed** | 42,814 (TN) | 7,997 (FP) |
| **Actual: Delayed** | 5,929 (FN) | 2,283 (TP) |

In [ ]:
# Load and display actual validation metrics
import json
from pathlib import Path

m = json.loads(Path('data/metrics/val_metrics.json').read_text())
vm = m['validation_metrics']
cm = m['confusion_matrix']
fi = m['feature_importances']

print(f'Model        : {m["model"]} ({m["label"]})')
print(f'Accuracy     : {vm["accuracy_pct"]:.2f}%')
print(f'Precision    : {vm["precision_pct"]:.2f}%')
print(f'Recall       : {vm["recall_pct"]:.2f}%')
print(f'F1-Score     : {vm["f1_pct"]:.2f}%')
print(f'ROC-AUC      : {vm["roc_auc_pct"]:.2f}%')
print()
print('Confusion Matrix:')
print(f'  TN={cm["TN"]:,}  FP={cm["FP"]:,}')
print(f'  FN={cm["FN"]:,}  TP={cm["TP"]:,}')
print()
print('Feature Importances (ranked):')
for feat, imp in sorted(fi.items(), key=lambda x: -x[1]):
    bar = '#' * int(imp * 100)
    print(f'  {feat:<22s} {imp:.4f}  {bar}')

In [ ]:
from IPython.display import Image, display
print('Confusion Matrix — Validation Set')
display(Image('reports/figures/cm_GBT_deeper_slower.png'))
print('Feature Importances — Selected Model')
display(Image('reports/figures/09_feature_importances_GBT_deeper_slower.png'))

## 5. Backend API

**Framework:** FastAPI 0.141.1 + Uvicorn 0.53.0  
**File:** `backend/main.py`

### Endpoints

| Method | Path | Description |
|---|---|---|
| GET | `/health` | Liveness check → `{"status": "ok"}` |
| POST | `/predict` | Returns delay prediction + probabilities |
| POST | `/analyse` | Returns grounded natural-language analysis |

Both ML artifacts are loaded **once at startup**:  
- `models/best_model.joblib` (2.2 MB — GradientBoostingClassifier)  
- `models/best_encoders.joblib` (10 KB — target-encoding mappings)

In [ ]:
# Test all three endpoints using FastAPI TestClient (no server needed)
import sys, json, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')
from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)

# ── /health ──────────────────────────────────────────────────────────────────
h = client.get('/health').json()
print('GET /health         :', h)

# ── /predict ─────────────────────────────────────────────────────────────────
inputs = {
    'op_unique_carrier': 'AA', 'origin': 'DFW', 'dest': 'ORD',
    'month': 7, 'day_of_month': 18, 'day_of_week': 5,
    'dep_hour': 17, 'arr_hour': 20,
    'crs_elapsed_time': 100.0, 'distance': 802.0,
}
pred = client.post('/predict', json=inputs).json()
print('POST /predict       :', pred['prediction'],
      f"(P(delayed)={pred['prob_delayed']:.4f})")

# ── /analyse ─────────────────────────────────────────────────────────────────
analysis = client.post('/analyse',
    json={'inputs': inputs, 'prediction': pred}).json()
print('POST /analyse verdict:', analysis['verdict_sentence'])
print('Risk factors count   :', len(analysis['risk_factors']))
print('Grounded in          :', analysis['data_sources'])

### Verified API Output — AA DFW → ORD sample

```
GET  /health   → {"status": "ok"}
POST /predict  → prediction: "delayed"  P(delayed)=0.821962
POST /analyse  → verdict: "The model predicts this flight is likely to be delayed
                 (82.2% probability of arriving 15 or more minutes late)."
```

**Start the server with:**
```bash
.\venv\Scripts\uvicorn.exe backend.main:app --host 127.0.0.1 --port 8000
```

## 6. Frontend Dashboard

**File:** `frontend/index.html` — single self-contained HTML file, no build step, no npm  
**Open:** Directly in any browser while the backend is running

### Features
- Form with 4 input groups: Route & Carrier, Schedule, Travel Date
- Native `<input type="time">` and `<input type="date">` pickers
- `dep_hour` / `arr_hour` extracted automatically from time pickers
- `month`, `day_of_month`, `day_of_week` extracted automatically from date picker
- Inline field-level validation with error messages
- Prediction result panel with colour coding and probability bars
- **"Analyse this flight"** button that calls `/analyse` and renders structured explanation
- Analyst panel with: Verdict, Risk Factors (each citing a source JSON file), Dataset Context, Model Performance note

## 7. Grounded GenAI Analyst

**File:** `backend/analyst.py`  
**Endpoint:** `POST /analyse`

### Design Principle
The analyst produces natural-language explanations that are **entirely grounded in pre-computed project outputs**. No external LLM, no API key, no hallucination risk.

| Output section | Grounded in |
|---|---|
| Verdict sentence | `/predict` probability from model |
| Risk factors — departure hour | `eda_stats.json: delay_rate_by_dep_hour` |
| Risk factors — month | `eda_stats.json: delay_rate_by_month` |
| Risk factors — carrier | `eda_stats.json: delay_rate_by_carrier` |
| Risk factors — origin | `eda_stats.json: delay_rate_by_top20_origin` |
| Feature rank references | `val_metrics.json: feature_importances` |
| Dataset context | `eda_stats.json: target_distribution` |
| Model performance note | `val_metrics.json: validation_metrics + confusion_matrix` |

### Sample Analysis — AA DFW→ORD, July 18, Friday, dep 17:xx

**Verdict:** The model predicts this flight is likely to be delayed (82.2% probability of arriving 15 or more minutes late).

**Risk factors (all numbers from `eda_stats.json`):**
- Departure time: 27.5% delay rate at afternoon departures (hour 17) — **above** overall 20.8%. Feature rank #2.
- Travel month: 29.6% delay rate in July — **above** overall 20.8%. Feature rank #3.
- Carrier (AA): 26.1% historical delay rate — **above** overall 20.8%. Feature rank #4.
- Origin (DFW): 27.4% delay rate departing DFW — **above** overall 20.8%. Feature rank #7.
- Day of week: Friday. Feature rank #8.

**Model note (from `val_metrics.json`):** GradientBoostingClassifier trained Jan–Sep, validated Oct–Nov (59,023 flights). ROC-AUC: 62.1%, Recall: 27.8%, Precision: 22.2%. Correctly identified 2,283 of 8,212 actual delayed flights.

## 8. Project Summary

| Component | Technology | File |
|---|---|---|
| Data processing | pandas, chunked I/O | `src/data/processing.py` |
| EDA | matplotlib, seaborn | `src/eda/eda.py` |
| ML experiment | scikit-learn | `src/ml/experiment.py` |
| ML baseline | scikit-learn | `src/ml/train.py` |
| Model artifact | GradientBoostingClassifier | `models/best_model.joblib` |
| Encoders | OOF target encoding | `models/best_encoders.joblib` |
| Backend API | FastAPI + Uvicorn | `backend/main.py` |
| Grounded analyst | Pure Python, JSON-grounded | `backend/analyst.py` |
| Frontend | HTML + CSS + vanilla JS | `frontend/index.html` |

### Final Model Performance
- **Model:** GradientBoostingClassifier (`GBT_deeper_slower`) — n=300, depth=6, lr=0.05
- **Encoding:** 5-fold OOF smoothed target encoding (leakage-safe, verified)
- **ROC-AUC:** 62.13% | **Recall:** 27.80% | **Precision:** 22.21% | **F1:** 24.69%
- **Test set (December 2024):** 29,242 rows — held out, not evaluated

### Limitations
- No real-time weather or NAS ground delay data — significant sources of flight delay are not in the pre-flight feature set
- The model is trained on 2024 data only; performance on different years may differ
- ROC-AUC of 62% reflects the difficulty of predicting delays using only scheduled information
- The December test set remains unseen and available for final unbiased evaluation